# MODULE 9 — HANDS-ON PROJECT
# Customer & Sales Analytics System

---

> **Scenario:**  
> You've just joined **TechMart** as a Senior Data Analyst.  
> The Head of Sales hands you a messy Excel dump from their CRM system.  
> Your task: clean it, compute KPIs, extract business insights, and produce a
> **dashboard-ready dataset** for the BI team by end of week.

**Deliverables:**
1. Clean dataset (gold-layer)
2. KPI summary table
3. Business insights report
4. Dashboard-ready aggregated tables
5. Anomaly detection

---

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 120)

print('Libraries loaded. Starting TechMart Analytics Project...')

### STEP 1 — Generate the Raw Messy Dataset
*This simulates the dirty CRM export you would receive from a real system.*

In [ ]:
np.random.seed(2024)
N = 2000  # 2,000 orders

# ── Raw CRM Export (as messy as real data gets) ───────────────────────────────
products_catalog = {
    'Laptop Pro'     : 1299, 'Laptop Basic': 699,
    'Smartphone X'   : 899,  'Smartphone Lite': 449,
    'Tablet Premium' : 799,  'Tablet Standard': 399,
    'Monitor 27"'    : 549,  'Monitor 24"': 349,
    'Mechanical Keyboard': 149, 'Wireless Mouse': 59,
    'USB Hub'        : 49,   'Webcam HD': 99
}

product_names    = list(products_catalog.keys())
product_prices   = list(products_catalog.values())
product_category = ['Computing','Computing','Mobile','Mobile','Computing',
                    'Computing','Computing','Computing','Peripheral',
                    'Peripheral','Peripheral','Peripheral']

# Customer base
n_customers = 300
first_names = ['Alice','Bob','Carol','David','Eve','Frank','Grace','Henry',
               'Ivy','Jack','Karen','Leo','Mia','Noah','Olivia','Paul',
               'Quinn','Rachel','Steve','Tina','Uma','Victor','Wendy','Xander']
last_names  = ['Smith','Johnson','Williams','Brown','Jones','Miller','Davis',
               'Wilson','Moore','Taylor','Anderson','Thomas','Jackson',
               'White','Harris','Martin','Thompson','Garcia','Martinez','Lee']

rng = np.random.default_rng(2024)
customer_pool = pd.DataFrame({
    'customer_id'  : range(1001, 1001 + n_customers),
    'customer_name': [f"{rng.choice(first_names)} {rng.choice(last_names)}" for _ in range(n_customers)],
    'email'        : [f"user{i}@{'gmail' if i%3==0 else 'yahoo' if i%3==1 else 'corp'}.com"
                      for i in range(n_customers)],
    'segment'      : rng.choice(['Enterprise', 'SMB', 'Consumer', 'Education'],
                                 n_customers, p=[0.15, 0.30, 0.40, 0.15]),
    'region'       : rng.choice(['North America', 'Europe', 'Asia Pacific', 'LATAM'],
                                 n_customers, p=[0.40, 0.30, 0.20, 0.10]),
    'account_manager': rng.choice(['Alice Chen','Bob Kumar','Carol White',
                                    'David Park','Eve Russo'], n_customers)
})

# Generate orders
order_customer_ids = rng.choice(customer_pool['customer_id'], N)
order_product_idx  = rng.choice(len(product_names), N)

raw_data = pd.DataFrame({
    'Order ID'        : [f'ORD-{100000+i}' for i in range(N)],
    'Customer ID'     : order_customer_ids,
    'Customer Name'   : [customer_pool.loc[customer_pool['customer_id']==cid,'customer_name'].values[0]
                         for cid in order_customer_ids],
    'Product'         : [product_names[i] for i in order_product_idx],
    'Category'        : [product_category[i] for i in order_product_idx],
    'Quantity'        : rng.integers(1, 8, N),
    'Unit Price'      : [product_prices[i] for i in order_product_idx],
    'Discount%'       : rng.choice([0, 5, 10, 15, 20, 25], N, p=[0.3,0.25,0.2,0.1,0.1,0.05]),
    'Order Date'      : [str(pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(d)))
                         for d in rng.integers(0, 730, N)],
    'Payment Method'  : rng.choice(['Credit Card','Bank Transfer','PayPal','Invoice'],
                                    N, p=[0.4,0.3,0.2,0.1]),
    'Ship Country'    : rng.choice(['US','USA','United States','UK','United Kingdom',
                                     'Germany','DE','France','Canada','CA','Japan','JP'],
                                    N, p=[0.15,0.10,0.10,0.05,0.05,0.10,0.05,
                                          0.05,0.05,0.05,0.10,0.10]),
    'Status'          : rng.choice(['Completed','Pending','Shipped','Cancelled','Returned'],
                                    N, p=[0.65, 0.10, 0.15, 0.05, 0.05]),
    'Notes'           : rng.choice(['', 'Urgent', 'Gift wrap', 'Corporate account',
                                     None, 'Reseller'], N)
})

# ── Inject realistic data quality issues ──────────────────────────────────────
# 1. Missing values
null_idx = rng.choice(N, 120, replace=False)
raw_data.loc[null_idx[:40], 'Unit Price']    = np.nan
raw_data.loc[null_idx[40:80], 'Order Date']  = np.nan
raw_data.loc[null_idx[80:], 'Payment Method'] = np.nan

# 2. Duplicate orders
dup_rows = raw_data.sample(15, random_state=42)
raw_data = pd.concat([raw_data, dup_rows], ignore_index=True)

# 3. Negative/zero quantities
raw_data.loc[rng.choice(N, 20, replace=False), 'Quantity'] = rng.choice([-1, 0], 20)

# 4. Price errors (outliers)
raw_data.loc[rng.choice(N, 5, replace=False), 'Unit Price'] = rng.choice([0.01, 99999], 5)

# 5. Mixed date formats
mixed_date_idx = rng.choice(N, 50, replace=False)
for idx in mixed_date_idx:
    dt = pd.Timestamp('2023-01-01') + pd.Timedelta(days=int(rng.integers(0, 730)))
    raw_data.loc[idx, 'Order Date'] = dt.strftime('%d/%m/%Y') if idx % 2 == 0 else dt.strftime('%B %d, %Y')

print(f'Raw dataset generated: {raw_data.shape[0]} rows, {raw_data.shape[1]} columns')
print(f'(Including {len(dup_rows)} duplicate rows and various data quality issues)')
raw_data.head(8)

### STEP 2 — Data Quality Audit (Before Cleaning)

In [ ]:
def full_audit(df):
    """Comprehensive data quality audit — run before any cleaning."""
    print('='*60)
    print('DATA QUALITY AUDIT REPORT')
    print('='*60)
    print(f'Rows    : {df.shape[0]:,}')
    print(f'Columns : {df.shape[1]}')
    print(f'Memory  : {df.memory_usage(deep=True).sum()/1024**2:.2f} MB')
    print(f'Full duplicates : {df.duplicated().sum()}')
    print()

    audit = pd.DataFrame({
        'dtype'       : df.dtypes,
        'non_null'    : df.notna().sum(),
        'null_count'  : df.isnull().sum(),
        'null_pct'    : (df.isnull().mean() * 100).round(1),
        'unique'      : df.nunique(),
        'top_value'   : [df[c].mode()[0] if df[c].notna().any() else 'N/A' for c in df.columns]
    })
    return audit

full_audit(raw_data)

### STEP 3 — Data Cleaning Pipeline

In [ ]:
def clean_techmart_data(raw):
    """
    Production-grade cleaning pipeline for TechMart CRM export.
    Returns: clean DataFrame ready for analysis
    """
    df = raw.copy()
    issues_log = []

    # ── Step 3.1: Standardize column names ────────────────────────────────────
    df.columns = (df.columns
                  .str.strip()
                  .str.lower()
                  .str.replace(' ', '_', regex=False)
                  .str.replace('%', '_pct', regex=False)
                  .str.replace('"', '', regex=False))
    issues_log.append('✓ Column names standardized')

    # ── Step 3.2: Remove full duplicates ──────────────────────────────────────
    n_before = len(df)
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    issues_log.append(f'✓ Removed {n_before - len(df)} duplicate rows')

    # ── Step 3.3: Fix Order Date ───────────────────────────────────────────────
    df['order_date'] = pd.to_datetime(df['order_date'],
                                      dayfirst=False,
                                      errors='coerce',
                                      infer_datetime_format=True)
    null_dates = df['order_date'].isnull().sum()
    # Impute with median date
    median_dt = df['order_date'].dropna().sort_values().iloc[len(df['order_date'].dropna())//2]
    df['order_date'].fillna(median_dt, inplace=True)
    issues_log.append(f'✓ Fixed {null_dates} unparseable order dates (imputed with median)')

    # ── Step 3.4: Fix Unit Price ───────────────────────────────────────────────
    # Fill missing prices with product median
    null_prices = df['unit_price'].isnull().sum()
    df['unit_price'] = df.groupby('product')['unit_price'].transform(
        lambda x: x.fillna(x.median())
    )
    # Remove price outliers (< $1 or > $50,000)
    price_outliers = ((df['unit_price'] < 1) | (df['unit_price'] > 50_000)).sum()
    df.loc[(df['unit_price'] < 1) | (df['unit_price'] > 50_000), 'unit_price'] = np.nan
    df['unit_price'] = df.groupby('product')['unit_price'].transform(
        lambda x: x.fillna(x.median())
    )
    issues_log.append(f'✓ Fixed {null_prices} missing prices, removed {price_outliers} price outliers')

    # ── Step 3.5: Fix invalid quantities ──────────────────────────────────────
    bad_qty = (df['quantity'] <= 0).sum()
    df = df[df['quantity'] > 0].copy()
    issues_log.append(f'✓ Removed {bad_qty} rows with invalid quantity (<= 0)')

    # ── Step 3.6: Standardize country names ───────────────────────────────────
    country_map = {
        'US': 'United States', 'USA': 'United States',
        'UK': 'United Kingdom',
        'DE': 'Germany',
        'CA': 'Canada',
        'JP': 'Japan'
    }
    df['ship_country'] = (df['ship_country']
                          .map(country_map)
                          .fillna(df['ship_country']))
    issues_log.append('✓ Country names normalized')

    # ── Step 3.7: Fill remaining nulls ────────────────────────────────────────
    df['payment_method'].fillna('Unknown', inplace=True)
    df['notes'].fillna('', inplace=True)
    issues_log.append('✓ Remaining nulls filled with sentinels')

    # ── Step 3.8: Compute revenue ─────────────────────────────────────────────
    df['gross_revenue'] = (df['quantity'] * df['unit_price']).round(2)
    df['net_revenue']   = (df['gross_revenue'] * (1 - df['discount__pct'] / 100)).round(2)
    df['discount_amount'] = (df['gross_revenue'] - df['net_revenue']).round(2)
    issues_log.append('✓ Revenue columns computed')

    # ── Step 3.9: Date features ───────────────────────────────────────────────
    df['order_year']    = df['order_date'].dt.year
    df['order_month']   = df['order_date'].dt.month
    df['order_quarter'] = df['order_date'].dt.quarter
    df['order_yearmon'] = df['order_date'].dt.to_period('M').astype(str)
    df['is_weekend']    = df['order_date'].dt.dayofweek >= 5

    # ── Step 3.10: Optimize dtypes ────────────────────────────────────────────
    cat_cols = ['product', 'category', 'ship_country', 'status',
                'payment_method', 'order_yearmon']
    for c in cat_cols:
        df[c] = df[c].astype('category')

    df['order_year']    = df['order_year'].astype('int16')
    df['order_month']   = df['order_month'].astype('int8')
    df['order_quarter'] = df['order_quarter'].astype('int8')

    print('\n'.join(issues_log))
    return df

# Run the cleaning pipeline
clean = clean_techmart_data(raw_data)
print(f'\nClean dataset: {clean.shape[0]:,} rows × {clean.shape[1]} columns')

In [ ]:
# Verify cleaning results
print('=== POST-CLEANING VALIDATION ===')
print(f'Nulls remaining: {clean.isnull().sum().sum()}')
print(f'Duplicate rows : {clean.duplicated().sum()}')
print(f'Invalid qty    : {(clean["quantity"] <= 0).sum()}')
print(f'Date range     : {clean["order_date"].min().date()} → {clean["order_date"].max().date()}')
print(f'Revenue range  : ${clean["net_revenue"].min():,.2f} → ${clean["net_revenue"].max():,.2f}')
print()
clean[['order_id', 'product', 'quantity', 'unit_price', 'discount__pct',
       'gross_revenue', 'net_revenue', 'order_date', 'status']].head(8)

### STEP 4 — KPI Calculation

In [ ]:
# ── Filter to completed/shipped orders only for revenue KPIs ──────────────────
active = clean[clean['status'].isin(['Completed', 'Shipped'])].copy()

# ── Core Business KPIs ────────────────────────────────────────────────────────
kpis = {
    '01. Total Orders'          : len(active),
    '02. Unique Customers'      : active['customer_id'].nunique(),
    '03. Gross Revenue ($)'     : active['gross_revenue'].sum(),
    '04. Net Revenue ($)'       : active['net_revenue'].sum(),
    '05. Total Discount ($)'    : active['discount_amount'].sum(),
    '06. Discount Rate (%)'     : (active['discount_amount'].sum() / active['gross_revenue'].sum() * 100),
    '07. Avg Order Value ($)'   : active['net_revenue'].mean(),
    '08. Median Order Value ($)': active['net_revenue'].median(),
    '09. Orders per Customer'   : len(active) / active['customer_id'].nunique(),
    '10. Revenue per Customer'  : active['net_revenue'].sum() / active['customer_id'].nunique(),
    '11. Units Sold'            : active['quantity'].sum(),
    '12. Avg Units per Order'   : active['quantity'].mean(),
    '13. Cancellation Rate (%)'  : (clean['status'] == 'Cancelled').mean() * 100,
    '14. Return Rate (%)'        : (clean['status'] == 'Returned').mean() * 100,
    '15. Weekend Order Share (%)': active['is_weekend'].mean() * 100
}

kpi_df = pd.Series(kpis).to_frame('value')
kpi_df['value'] = kpi_df['value'].round(2)
print('='*50)
print('TECHMART — EXECUTIVE KPI DASHBOARD')
print('='*50)
for k, v in kpis.items():
    if '$' in k:
        print(f'{k:<35}: ${v:>12,.0f}')
    elif '%' in k:
        print(f'{k:<35}: {v:>11.1f}%')
    else:
        print(f'{k:<35}: {v:>12,.1f}')

### STEP 5 — Business Insights Extraction

In [ ]:
# ── Insight 1: Product Performance Matrix ─────────────────────────────────────
product_perf = active.groupby('product', observed=True).agg(
    orders       = ('order_id', 'count'),
    units_sold   = ('quantity', 'sum'),
    gross_rev    = ('gross_revenue', 'sum'),
    net_rev      = ('net_revenue', 'sum'),
    discount_amt = ('discount_amount', 'sum'),
    avg_discount = ('discount__pct', 'mean')
).round(2)

product_perf['revenue_share_%'] = (product_perf['net_rev'] / product_perf['net_rev'].sum() * 100).round(1)
product_perf['avg_order_val']   = (product_perf['net_rev'] / product_perf['orders']).round(0)
product_perf = product_perf.sort_values('net_rev', ascending=False)

print('=== PRODUCT PERFORMANCE MATRIX ===')
print(product_perf[['orders','units_sold','net_rev','revenue_share_%',
                     'avg_order_val','avg_discount']].to_string())

In [ ]:
# ── Insight 2: Monthly Revenue Trend ──────────────────────────────────────────
monthly_trend = active.groupby('order_yearmon', observed=True).agg(
    orders   = ('order_id', 'count'),
    net_rev  = ('net_revenue', 'sum'),
    customers = ('customer_id', 'nunique')
).reset_index()

monthly_trend['rev_mom_pct'] = monthly_trend['net_rev'].pct_change() * 100
monthly_trend['rev_cumsum']  = monthly_trend['net_rev'].cumsum()
monthly_trend['arpu']        = (monthly_trend['net_rev'] / monthly_trend['customers']).round(0)

print('=== MONTHLY REVENUE TREND ===')
print(monthly_trend.set_index('order_yearmon').round(1).to_string())

In [ ]:
# ── Insight 3: Customer Segmentation (RFM Analysis) ──────────────────────────
# RFM = Recency, Frequency, Monetary — the gold standard of customer segmentation
snapshot_date = active['order_date'].max() + pd.Timedelta(days=1)

rfm = active.groupby('customer_id').agg(
    recency   = ('order_date',   lambda x: (snapshot_date - x.max()).days),
    frequency = ('order_id',     'count'),
    monetary  = ('net_revenue',  'sum')
).round(2)

# Score each dimension 1-5
rfm['R'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)  # lower=better
rfm['F'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'].astype(str) + rfm['F'].astype(str) + rfm['M'].astype(str)
rfm['RFM_Total'] = rfm[['R','F','M']].sum(axis=1)

# Segment customers
def rfm_segment(row):
    if row['RFM_Total'] >= 13: return 'Champions'
    elif row['RFM_Total'] >= 10: return 'Loyal Customers'
    elif row['RFM_Total'] >= 7:  return 'Potential Loyalists'
    elif row['RFM_Total'] >= 5:  return 'At Risk'
    else: return 'Lost'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)

seg_summary = rfm.groupby('segment').agg(
    customer_count = ('recency', 'count'),
    avg_recency    = ('recency', 'mean'),
    avg_frequency  = ('frequency', 'mean'),
    avg_monetary   = ('monetary', 'mean'),
    total_revenue  = ('monetary', 'sum')
).round(1).sort_values('total_revenue', ascending=False)

print('=== RFM CUSTOMER SEGMENTATION ===')
print(seg_summary.to_string())

In [ ]:
# ── Insight 4: Top 10% vs Bottom 10% Customers (Pareto Analysis) ─────────────
customer_revenue = active.groupby('customer_id')['net_revenue'].sum().sort_values(ascending=False)
total_rev = customer_revenue.sum()

top_10_pct_n    = max(1, int(len(customer_revenue) * 0.10))
top10_rev       = customer_revenue.head(top_10_pct_n).sum()
top10_share     = top10_rev / total_rev * 100

print('=== PARETO ANALYSIS ===')
print(f'Total customers : {len(customer_revenue):,}')
print(f'Top 10% = {top_10_pct_n} customers → ${top10_rev:,.0f} revenue ({top10_share:.1f}% of total)')
print(f'\nTop 10 customers by revenue:')
print(customer_revenue.head(10).apply(lambda x: f'${x:,.0f}').to_string())

In [ ]:
# ── Insight 5: Payment Method & Discount Analysis ─────────────────────────────
payment_analysis = active.groupby('payment_method', observed=True).agg(
    orders      = ('order_id', 'count'),
    net_rev     = ('net_revenue', 'sum'),
    avg_disc    = ('discount__pct', 'mean'),
    avg_qty     = ('quantity', 'mean')
).round(2)
payment_analysis['share_%'] = (payment_analysis['net_rev'] / payment_analysis['net_rev'].sum() * 100).round(1)
payment_analysis = payment_analysis.sort_values('net_rev', ascending=False)
print('=== PAYMENT METHOD ANALYSIS ===')
print(payment_analysis.to_string())

In [ ]:
# ── Insight 6: Country Performance ────────────────────────────────────────────
country_perf = active.groupby('ship_country', observed=True).agg(
    orders    = ('order_id', 'count'),
    customers = ('customer_id', 'nunique'),
    net_rev   = ('net_revenue', 'sum'),
    avg_order = ('net_revenue', 'mean')
).round(2)
country_perf['rev_share_%'] = (country_perf['net_rev'] / country_perf['net_rev'].sum() * 100).round(1)
country_perf = country_perf.sort_values('net_rev', ascending=False)
print('=== COUNTRY REVENUE BREAKDOWN ===')
print(country_perf.to_string())

In [ ]:
# ── Insight 7: Anomaly Detection ──────────────────────────────────────────────
print('=== ANOMALY DETECTION ===')

# 1. Revenue anomalies using Z-score
active2 = active.copy()
mean_rev = active2['net_revenue'].mean()
std_rev  = active2['net_revenue'].std()
active2['rev_zscore'] = ((active2['net_revenue'] - mean_rev) / std_rev).round(2)
anomalies = active2[active2['rev_zscore'].abs() > 3]
print(f'Revenue outlier orders (|Z| > 3): {len(anomalies)}')
if len(anomalies) > 0:
    print(anomalies[['order_id', 'product', 'quantity', 'net_revenue', 'rev_zscore']].to_string(index=False))

# 2. Single customer ordering huge volumes
cust_orders = active.groupby('customer_id')['order_id'].count()
Q3 = cust_orders.quantile(0.75)
IQR = Q3 - cust_orders.quantile(0.25)
whale_customers = cust_orders[cust_orders > Q3 + 3 * IQR]
print(f'\nWhale customers (extreme order count): {len(whale_customers)}')
print(whale_customers.head())

# 3. Days with zero orders
date_range = pd.date_range(active['order_date'].min(), active['order_date'].max(), freq='D')
active_dates = active['order_date'].dt.normalize()
missing_days = date_range[~date_range.isin(active_dates)]
print(f'\nDays with zero orders: {len(missing_days)}')

### STEP 6 — Dashboard-Ready Dataset Creation

In [ ]:
# ── 6A: Monthly KPI Table (for line charts) ───────────────────────────────────
dash_monthly = active.groupby('order_yearmon', observed=True).agg(
    orders          = ('order_id', 'count'),
    unique_customers= ('customer_id', 'nunique'),
    gross_revenue   = ('gross_revenue', 'sum'),
    net_revenue     = ('net_revenue', 'sum'),
    discount_total  = ('discount_amount', 'sum'),
    units_sold      = ('quantity', 'sum'),
    avg_order_value = ('net_revenue', 'mean')
).round(2).reset_index()

dash_monthly['discount_rate']  = (dash_monthly['discount_total'] / dash_monthly['gross_revenue'] * 100).round(2)
dash_monthly['rev_mom_pct']    = dash_monthly['net_revenue'].pct_change().mul(100).round(1)
dash_monthly['orders_mom_pct'] = dash_monthly['orders'].pct_change().mul(100).round(1)

print('Monthly Dashboard Table (first 6 months):')
print(dash_monthly.head(6).to_string(index=False))

In [ ]:
# ── 6B: Product × Category × Month Pivot (for BI tools) ──────────────────────
dash_product = active.groupby(['order_yearmon', 'category', 'product'], observed=True).agg(
    orders    = ('order_id', 'count'),
    net_rev   = ('net_revenue', 'sum'),
    units     = ('quantity', 'sum')
).round(2).reset_index()

print('Product Performance Table shape:', dash_product.shape)
print(dash_product.head(8).to_string(index=False))

In [ ]:
# ── 6C: Customer 360 Table (one row per customer) ─────────────────────────────
customer_360 = active.groupby('customer_id').agg(
    total_orders    = ('order_id', 'count'),
    total_revenue   = ('net_revenue', 'sum'),
    avg_order_value = ('net_revenue', 'mean'),
    first_order     = ('order_date', 'min'),
    last_order      = ('order_date', 'max'),
    favourite_product = ('product', lambda x: x.mode()[0]),
    countries_ordered = ('ship_country', 'nunique'),
    avg_discount    = ('discount__pct', 'mean')
).round(2).reset_index()

customer_360['customer_lifetime_days'] = (
    customer_360['last_order'] - customer_360['first_order']
).dt.days

customer_360['recency_days'] = (
    snapshot_date - customer_360['last_order']
).dt.days

# Merge RFM segment
customer_360 = customer_360.merge(
    rfm[['segment']].reset_index(),
    on='customer_id', how='left'
)

print('Customer 360 Table shape:', customer_360.shape)
print(customer_360.head(5).to_string(index=False))

In [ ]:
# ── Save all dashboard tables ─────────────────────────────────────────────────
import os

output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Save to Parquet (for BI tools like Power BI / Tableau)
clean.to_parquet(f'{output_dir}/techmart_gold_layer.parquet', index=False)
dash_monthly.to_parquet(f'{output_dir}/dashboard_monthly.parquet', index=False)
dash_product.to_parquet(f'{output_dir}/dashboard_product.parquet', index=False)
customer_360.to_parquet(f'{output_dir}/customer_360.parquet', index=False)
rfm.reset_index().to_parquet(f'{output_dir}/rfm_segments.parquet', index=False)

# Also save to CSV for sharing with non-technical stakeholders
dash_monthly.to_csv(f'{output_dir}/dashboard_monthly.csv', index=False)
customer_360.to_csv(f'{output_dir}/customer_360.csv', index=False)
kpi_df.to_csv(f'{output_dir}/kpi_summary.csv')

print('All output files saved to /output/')
for f in os.listdir(output_dir):
    size = os.path.getsize(f'{output_dir}/{f}')
    print(f'  {f:<40} {size/1024:.1f} KB')

### STEP 7 — Executive Summary
*This is the narrative you would write in your stakeholder report.*

In [ ]:
total_net = active['net_revenue'].sum()
top_product = product_perf['net_rev'].idxmax()
top_country = country_perf['net_rev'].idxmax()
champions_rev = seg_summary.loc['Champions', 'total_revenue'] if 'Champions' in seg_summary.index else 0
champions_count = seg_summary.loc['Champions', 'customer_count'] if 'Champions' in seg_summary.index else 0

print(f"""
╔══════════════════════════════════════════════════════════════╗
║          TECHMART ANALYTICS — EXECUTIVE SUMMARY             ║
╚══════════════════════════════════════════════════════════════╝

BUSINESS PERFORMANCE
  • Total Net Revenue    : ${total_net:>12,.0f}
  • Active Orders        : {len(active):>12,}
  • Unique Customers     : {active['customer_id'].nunique():>12,}
  • Avg Order Value      : ${active['net_revenue'].mean():>12,.0f}

TOP PERFORMERS
  • Best Product         : {top_product}
  • Best Country         : {top_country}
  • Champion Customers   : {int(champions_count)} customers generating 
                           ${champions_rev:,.0f} revenue

DATA QUALITY ACTIONS TAKEN
  • Removed 15 duplicate orders
  • Fixed 120 missing values across 3 columns
  • Standardized country names (12 → 6 unique)
  • Removed invalid quantities (negative/zero)
  • Capped price outliers via product-median imputation

RECOMMENDED ACTIONS
  1. Re-engage 'At Risk' and 'Lost' customer segments
  2. Increase inventory for top product line
  3. Investigate cancellation rates in high-discount orders
  4. Expand presence in {top_country} (highest revenue market)
""")